In [1]:
"""
The Goal of this notebook:
- update the reaction wheel safety constraints
 to stay within orientation limits


Verification:

- show video with max torque cmd
- satellite should stay within movement constraints
- print verification message when torque is limited to console
- run scenario with max torque
- run scenario with half torque

API:
-inputs:
use from existing RW reaction wheel safety mechanism

-output:
None (effect is limiting reaction wheel torque)

use defaults:
thresholds are in off nadir orientation:
should always be stopped (omega_sat = 0) and return to safe:
- 45 deg off nadir
reduce torque gradually  before safe mode kicks in
- start 5 deg before safe mode activation (dynamic based on omega_sat)
-> from the current rot speed calculate
minimal deceleration distance with max anti-torque

example: sat spins at 3deg/s then the braking distance is 24.6deg
-> safe mode start = 45deg -24.6deg = 20.4 deg
(so theoretically if safe mode works correctly you can never reach 3deg/s)

if safe mode is activated the safe controller makes this maneuver:
    -brake relative spin rate toward orbit track
    -cruise toward nadir at 0.35 deg/s (instantaneous nadir + orbit feedforward)
    -settle on nadir
    -lockout 10s: agent cut, OBC nadir hold continues


"""

'\nThe Goal of this notebook:\n- update the reaction wheel safety constraints\n to stay within orientation limits\n\n\nVerification:\n\n- show video with max torque cmd\n- satellite should stay within movement constraints\n- print verification message when torque is limited to console\n- run scenario with max torque\n- run scenario with half torque\n\nAPI:\n-inputs:\nuse from existing RW reaction wheel safety mechanism\n\n-output:\nNone (effect is limiting reaction wheel torque)\n\nuse defaults:\nthresholds are in off nadir orientation:\nshould always be stopped (omega_sat = 0) and return to safe:\n- 45 deg off nadir\nreduce torque gradually  before safe mode kicks in\n- start 5 deg before safe mode activation (dynamic based on omega_sat)\n-> from the current rot speed calculate\nminimal deceleration distance with max anti-torque\n\nexample: sat spins at 3deg/s then the braking distance is 24.6deg\n-> safe mode start = 45deg -24.6deg = 20.4 deg\n(so theoretically if safe mode works cor

In [2]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
backend_root = notebook_dir
for _ in range(6):
    if (backend_root / "simulation").is_dir():
        break
    backend_root = backend_root.parent
os.chdir(backend_root)
sys.path.insert(0, str(backend_root))
_s01_dir = backend_root / "notebooks" / "s01"
sys.path.insert(0, str(_s01_dir))
print(f"backend_root={backend_root}")

backend_root=/home/cedric/code/auto-sat-control/backend


In [3]:
from importlib import reload

import environment_definition.constants.ATTITUDE_SAFETY as attitude_safety
import simulation.attitude_controller as attitude_controller
import s01_utils.movement_constraints_patch as movement_constraints_patch

# Reload constants before dependents (reload() does not refresh imported names in sys.modules).
reload(attitude_safety)
reload(attitude_controller)
reload(movement_constraints_patch)

from environment_definition.constants.SATELLITE import MOMENT_OF_INERTIA_2D, REACTION_WHEEL_MAX_TORQUE
from environment_definition.constants.UNIT_REGISTRY import UREG as ureg
from simulation.attitude_controller import braking_distance_rad, safe_mode_activation_angle_rad
from s01_utils.movement_constraints_patch import (
    build_fast_movement_setup,
    build_movement_stepper,
    build_movement_video_setup,
    export_movement_verification_video,
    make_delayed_max_policy,
    max_off_nadir_deg_series,
    movement_simulation_config,
    print_movement_verification_summary,
    run_policy_rollout,
)
from utils.notebook.video import init_video_cell

init_video_cell()

In [4]:
from environment_definition.constants.ATTITUDE_SAFETY import OFF_NADIR_HARD_LIMIT_DEG

# Notebook 03 worked example (ω = 3°/s); exact values also locked in test_attitude_controller.py
omega = 3.0 * ureg.deg / ureg.s
doc_brake = 24.6 * ureg.deg
doc_arm = 20.4 * ureg.deg
tol = 1.0 * ureg.deg

brake = (
    braking_distance_rad(
        omega_sat=omega,
        tau_max=REACTION_WHEEL_MAX_TORQUE,
        sat_inertia=MOMENT_OF_INERTIA_2D,
    )
    * ureg.rad
).to(ureg.deg)
arm = (
    safe_mode_activation_angle_rad(
        off_nadir_limit=OFF_NADIR_HARD_LIMIT_DEG,
        omega_sat=omega,
        tau_max=REACTION_WHEEL_MAX_TORQUE,
        sat_inertia=MOMENT_OF_INERTIA_2D,
    )
    * ureg.rad
).to(ureg.deg)

print(f"braking_distance: {brake:~}  (doc example {doc_brake:~})")
print(f"safe_mode_arm:      {arm:~}  (doc example {doc_arm:~})")

# Useful: θ_arm = θ_hard − θ_brake (definition used by the controller)
assert abs((arm - (OFF_NADIR_HARD_LIMIT_DEG - brake)).to(ureg.deg).magnitude) < 0.05

# Smoke: docstring numbers still match SATELLITE constants after edits (re-run pytest for full lock)
assert (brake - doc_brake).to(ureg.deg).magnitude < tol.to(ureg.deg).magnitude
assert (arm - doc_arm).to(ureg.deg).magnitude < tol.to(ureg.deg).magnitude

braking_distance: 24.504422698000386 deg  (doc example 24.6 deg)
safe_mode_arm:      20.495577301999614 deg  (doc example 20.4 deg)


In [5]:
setup = build_fast_movement_setup(seed=0, include_cameras=False)
sim_cfg = movement_simulation_config(attitude_controller_enabled=True)
stepper, _ = build_movement_stepper(setup, simulation_config=sim_cfg, patch=True)
policy = make_delayed_max_policy(stepper, delay_s=5.0, tau_scale=1.0)
series_max, events_max = run_policy_rollout(stepper, policy, print_events=True)
summary_max = print_movement_verification_summary(series_max, events_max, delay_s=5.0)

SAFE_MODE_INTERVAL_BRAKE step=53 t=21.1s off_nadir=22.6deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=54 t=21.5s off_nadir=23.8deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=55 t=21.9s off_nadir=24.8deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=56 t=22.3s off_nadir=25.9deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=57 t=22.7s off_nadir=26.9deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=58 t=23.1s off_nadir=27.9deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=59 t=23.5s off_nadir=28.9deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=60 t=23.9s off_nadir=29.8deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=61 t=24.3s off_nadir=30.7deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=62 t=24.7s off_nadir=31.6deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=63 t=25.1s off_nadir=32.5deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=64 t=25.5s off_nadir=33.3deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=65 t=25.9s

/home/cedric/code/auto-sat-control/backend/simulation/stepper.py:164: UserWarning: controller_update_interval (1 s) is not an integer multiple of simulation_timestep (0.4 s); using nearest multiple: 0.8 s (2 sim steps).
  self._controller_interval_steps, self._effective_controller_interval_s = resolve_controller_interval_steps(


SAFE_MODE_INTERVAL_BRAKE step=74 t=29.5s off_nadir=39.9deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=75 t=29.9s off_nadir=40.4deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=76 t=30.3s off_nadir=40.9deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=77 t=30.7s off_nadir=41.3deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=78 t=31.1s off_nadir=41.7deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=79 t=31.5s off_nadir=42.1deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=80 t=31.9s off_nadir=42.5deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=81 t=32.3s off_nadir=42.8deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=82 t=32.7s off_nadir=43.1deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=83 t=33.1s off_nadir=43.4deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=84 t=33.5s off_nadir=43.6deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=85 t=33.9s off_nadir=43.8deg cmd=0.100 out=-0.100
SAFE_MODE_INTERVAL_BRAKE step=86 t=34.3s

In [6]:
setup_half = build_fast_movement_setup(seed=0, include_cameras=False)
stepper_half, _ = build_movement_stepper(setup_half, simulation_config=sim_cfg, patch=True)
policy_half = make_delayed_max_policy(stepper_half, delay_s=5.0, tau_scale=0.5)
series_half, events_half = run_policy_rollout(stepper_half, policy_half, print_events=False)
summary_half = print_movement_verification_summary(series_half, events_half, delay_s=5.0)

Movement verification summary
  sim_total_s:     133.7
  t_coast_end:     5.0s
  max_off_nadir:   44.96 deg
  final_off_nadir: 0.00 deg
  lockout_off_nadir: 0.00 deg
  n_warnings:      0
  n_takeover:      1
  n_lockout_steps: 0
  n_safe_exit:     0
  takeover:        step=79 t=31.5s
FAIL: envelope_ok=True takeover_ok=True lockout_ok=False recovery_ok=True


In [7]:
import numpy as np

off_max = max_off_nadir_deg_series(series_max)
assert float(np.max(off_max)) <= 45.5, f"max off-nadir {float(np.max(off_max)):.2f} deg"
assert summary_max["takeover_ok"]
# Determinism: repeat takeover step
stepper_b, _ = build_movement_stepper(setup, simulation_config=sim_cfg, patch=True)
_, events_b = run_policy_rollout(
    stepper_b, make_delayed_max_policy(stepper_b, delay_s=5.0, tau_scale=1.0), print_events=False
)
t0 = [e for e in events_max if e["event"] == "SAFE_MODE_TAKEOVER"][0]["step"]
t1 = [e for e in events_b if e["event"] == "SAFE_MODE_TAKEOVER"][0]["step"]
assert t0 == t1
print(f"deterministic takeover step={t0}")

deterministic takeover step=53


In [ ]:
import matplotlib

matplotlib.use("Agg")

video_setup = build_movement_video_setup(seed=0)
stepper_vid, _ = build_movement_stepper(video_setup, simulation_config=sim_cfg, patch=True)
policy_vid = make_delayed_max_policy(stepper_vid, delay_s=5.0, tau_scale=1.0)
series_vid, _ = run_policy_rollout(stepper_vid, policy_vid, print_events=False, show_progress=True)
video_path = export_movement_verification_video(series_vid, play=True)


movement rollout: 100%|██████████| 335/335 [00:01<00:00, 234.37it/s]


[video] archived previous export -> /home/cedric/code/auto-sat-control/backend/notebooks/s01/artifacts/video_archive/002-movement_constraints_delayed_max.mp4


Writing video: 100%|██████████| 150/150 [00:12<00:00, 12.16frame/s]

[mpo_video:after_export] movement_constraints_delayed_max.mp4 (316202 bytes, codec=h264)
[mpo_video:play] movement_constraints_delayed_max.mp4 (316202 bytes, codec=h264)


artifact=/home/cedric/code/auto-sat-control/backend/notebooks/s01/artifacts/movement_constraints_delayed_max.mp4


In [9]:
video_path

PosixPath('/home/cedric/code/auto-sat-control/backend/notebooks/s01/artifacts/movement_constraints_delayed_max.mp4')

## Human sign-off (4b)

- [ ] **Coast (~5 s):** satellite holds nadir before torque step
- [ ] **Violation (~5 s):** agent requests full torque; attitude slews
- [ ] **Takeover / recovery:** safe-mode recovery visible; off-nadir returns toward nadir
- [ ] **Artifact:** `backend/notebooks/s01/artifacts/movement_constraints_delayed_max.mp4`

Visual verification per `.cursor/skills/visual-output-verification/SKILL.md` — confirm timeline matches console takeover step.